In [1]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')                               # project/notebooks -> project
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))                # so `from src....` imports work
print('working from:', ROOT.name)


working from: project


# Project Pipeline

Stage 07. **I based it on the lecture notebook.**

## Install Missing Packages

In [2]:
# Install missing packages (uncomment and run to install).
# !pip install pandas python-dotenv pyarrow requests

## Load and Config Environment

In [3]:
from pathlib import Path

# Project root.
ROOT = Path.cwd()
if not (ROOT/".env.example").exists() and (ROOT.parent/".env.example").exists():
    ROOT = ROOT.parent

CHECKS = [
    ("src/config.py", "NEEDED", "env helpers"),
    ("src/utils.py", "NEEDED", "summary stats"),
    ("src/io_utils.py", "NEEDED", "save and load helpers"),
    ("src/cleaning.py", "NEEDED", "Stage 06 cleaning helpers"),
    ("src/outliers.py", "NEEDED", "Stage 07 outlier helpers"),
    ("data/raw/prismatic_evoluations_prices.csv", "NEEDED", "Prismatic Evolutions prices"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT/rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    raise FileNotFoundError(f"{missing} needed file(s) missing under {ROOT}")
print("\nAll needed files present.")

Looking in: /Users/ghostof0days/projects/bootcamp/project

  [OK ]  NEEDED    src/config.py                       env helpers
  [OK ]  NEEDED    src/utils.py                        summary stats
  [OK ]  NEEDED    src/io_utils.py                     save and load helpers
  [OK ]  NEEDED    src/cleaning.py                     Stage 06 cleaning helpers
  [OK ]  NEEDED    src/outliers.py                     Stage 07 outlier helpers
  [OK ]  NEEDED    data/raw/prismatic_evoluations_prices.csv  Prismatic Evolutions prices

All needed files present.


In [4]:
import sys

import pandas as pd
from dotenv import load_dotenv

# Import helpers from src/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import drop_missing, fill_missing_median, normalize_data
from src.config import get_key
from src.io_utils import read_df, save_csv, validate, write_df
from src.outliers import detect_outliers_iqr, winsorize_series
from src.utils import get_summary_stats

# Load `.env`.
load_dotenv(ROOT/".env")

RAW = ROOT/(get_key("DATA_DIR_RAW", "data/raw") or "data/raw")
PROC = ROOT/(get_key("DATA_DIR_PROCESSED", "data/processed") or "data/processed")
print("RAW:", RAW.resolve())
print("PROC:", PROC.resolve())

RAW: /Users/ghostof0days/projects/bootcamp/project/data/raw
PROC: /Users/ghostof0days/projects/bootcamp/project/data/processed


## Acquire raw TCGCSV snapshot into data/raw/


In [5]:
import requests

# Prismatic Evolutions live snapshot. Archive history stays in prismatic_evoluations_prices.csv.
GROUP_ID = 23821
TCGCSV_BASE = f"https://tcgcsv.com/tcgplayer/3/{GROUP_ID}"
headers = {"User-Agent": "Pokemon-Set-Screener/1.0"}
prices_response = requests.get(f"{TCGCSV_BASE}/prices", headers=headers, timeout=30)
prices_response.raise_for_status()
prices = pd.DataFrame(prices_response.json()["results"])
products_response = requests.get(f"{TCGCSV_BASE}/products", headers=headers, timeout=30)
products_response.raise_for_status()
products = pd.DataFrame(products_response.json()["results"])
snapshot = prices.merge(products[["productId", "name"]], on="productId", how="left")
snapshot = snapshot.rename(columns={"name": "card_name", "marketPrice": "market_price"})
snapshot["market_price"] = pd.to_numeric(snapshot["market_price"], errors="coerce")
print(validate(snapshot, ["productId", "card_name", "market_price"]))
save_csv(snapshot, prefix="api", raw_dir=RAW, source="tcgcsv", set="prismatic-evolutions")


{'missing': [], 'shape': (498, 8), 'na_total': 170}
Saved: /Users/ghostof0days/projects/bootcamp/project/data/raw/api_source-tcgcsv_set-prismatic-evolutions_20260820-123242.csv


PosixPath('/Users/ghostof0days/projects/bootcamp/project/data/raw/api_source-tcgcsv_set-prismatic-evolutions_20260820-123242.csv')

## Load, validate, and summarize CSV data

In [6]:
card_prices = pd.read_csv(RAW/"prismatic_evoluations_prices.csv", parse_dates=["date"])
print(validate(card_prices, ["date", "card_name", "market_price"]))
get_summary_stats(card_prices)

{'missing': [], 'shape': (61279, 4), 'na_total': 0}
[2026-08-20 12:32:42.502802] Function 'get_summary_stats' called.


,market_price
count,61279.000000
mean,51.546560
std,141.875164
min,2.000000
25%,4.980000
50%,10.100000
75%,34.595000
max,1618.750000


## Clean and save CSV prices (fill in missing market prices as well as only take valid price). For this, I define valid as positive.

In [7]:
positive = card_prices[card_prices["market_price"] > 0].copy()
cleaned = normalize_data(
    drop_missing(fill_missing_median(positive, ["market_price"]), threshold=0.5),
    ["market_price"],
)
cleaned.to_csv(PROC/"prismatic_prices_cleaned.csv", index=False)
print("Cleaned shape:", cleaned.shape)

Cleaned shape: (61279, 4)


## Save and load the cleaned prices as a Parquet file.

In [8]:
parquet_path = PROC/"prismatic_prices_cleaned.parquet"
write_df(cleaned, parquet_path)
parquet_prices = read_df(parquet_path)
print("Parquet shape:", parquet_prices.shape)

Parquet shape: (61279, 4)


## Flag IQR outliers on market price and compare treatments

In [9]:
# drop non-positive prices.
positive = card_prices[card_prices["market_price"] > 0].copy()
# k=0.5 (k=1.5 from example still flags about 0.13).
positive["outlier_iqr"] = detect_outliers_iqr(positive["market_price"], k=0.5)
iqr_outlier_fraction = positive["outlier_iqr"].mean()
print("IQR outlier fraction:", iqr_outlier_fraction)

# ~ flips the iqr mask so we keep non-outlier rows.
summary_all = positive["market_price"].describe()[["mean", "50%", "std"]].rename({"50%": "median"})
summary_outliers_dropped = positive.loc[~positive["outlier_iqr"], "market_price"].describe()[["mean", "50%", "std"]].rename({"50%": "median"})
winsorized = winsorize_series(positive["market_price"])
summary_winsorized = winsorized.describe()[["mean", "50%", "std"]].rename({"50%": "median"})
comparison = pd.concat({"all": summary_all, "outliers_dropped": summary_outliers_dropped, "winsorized": summary_winsorized}, axis=1)
print("Summary statistics:")
comparison

IQR outlier fraction: 0.18942867866642732
Summary statistics:


,all,outliers_dropped,winsorized
mean,51.546560,12.326210,38.058473
median,10.100000,7.830000,10.100000
std,141.875164,11.375027,62.820303
